# LQR Solver

This notebook implements an optimal control solver for a linear time-invariant system as defined by $\dot x=Ax+Bu.$

The infinite-0horizon cost function for the optimization is $J=\int^\infty_0 [x^T Q x + u^T R u] dt$

The optimal cost-to-go function $J^*=x^T S x$ is found by optimizing $f(x, u)=x^T Q x + u^T R u + \frac{\partial J^*}{\partial x}(Ax + Bu)$ as $\forall x, 0=\underset{u}{min} f(x, u)$

Since $\frac{\partial J^*}{\partial x}=2 x^T S$, the function to be optimized becomes $f(x, u)=x^T Q x + u^T R u + 2 x^T SA x + 2 x^T SB u$. Optimizing w. r. t. $u$ and since all terms are quadratic, the minimum can be found by $\frac{\partial f}{\partial u}=0$, which yields:

$2 u^T R + 2 x^T SB = 0$ resulting in an optimal control policy by solving the matrix equation:

$u^T R = -x^T SB$

$u^T = -x^T SBR^{-1}$

$u = (-x^T SBR^{-1})^T=-R^{-1}B^TSx=-Kx\implies K=R^{-1}B^TS$

Substituting this back into the optimization problem, yields the matrix equation to be solved for S and, subsequently, K: 

$0=x^T Q x - x^T SBR^{-1} R (-R^{-1}B^TSx) + 2 x^T SA x + 2 x^T SB (-R^{-1}B^TSx)$ 

$0=x^T[Q + 2 SA - SB R^{-1}B^TS]x$ 

$0=x^T[Q + SA + A^TS - SB R^{-1}B^TS]x, \forall x$ 

$\implies Q + SA + A^TS - SBR^{-1}B^TS=0$

In [ ]:
try:
    import jax.numpy as jnp
    import time
    import math
    from pydrake.all import LinearQuadraticRegulator
    print('Imported all libraries successfully.')
except Exception as e:
    print('Unable to import libraries.')
    raise e

In [ ]:
def calculate_result(A, B, Q, R, S):
    return (Q + S @ A + A.T @ S - S @ B @ R**(-1) @ B.T @ S)
def calculate_accuracy(A, B, Q, R, S): 
    return jnp.sum(calculate_result(A, B, Q, R, S)**2)
def deriveK(S, R, B):
    return jnp.linalg.inv(R) @ B.T @ S

## GA Approach

The DNA is S and the fitness function is simply $\sum (Q + SA + A^TS - SBR^{-1}B^TS)^2$.

In [ ]:
import numpy as np

def mutate(M, mutation_range, rng_instance):
    return M + (rng_instance.random(size=(M.shape[0], M.shape[1])) * mutation_range - mutation_range / 2.0)

def ensure_spd(M, min_eig=1e-6):
    """Ensure matrix is symmetric positive definite."""
    M_sym = 0.5 * (M + M.T)
    eigvals = jnp.linalg.eigvalsh(M_sym)
    min_eigenvalue = jnp.min(eigvals)
    if min_eigenvalue < min_eig:
        M_sym = M_sym + (min_eig - min_eigenvalue) * jnp.eye(M.shape[0])
    return M_sym

def solve_lqr_ga(A, B, Q, R, 
                 population_size=512,
                 generations=4000,
                 variability=0.2,
                 elite_fraction=0.1,
                 tournament_size=4):
    """
    Solve LQR using genetic algorithm with SPD constraint and diversity preservation. 
    """
    if len(A.shape) != 2 or A.shape[0] != A.shape[1]:
        raise ValueError('A should be a square matrix.')
    
    n = A.shape[0]
    rng = np.random.default_rng(seed=42)
    
    max_val = jnp.max(A + B + Q + R)
    min_val = jnp.min(A + B + Q + R)
    value_range = max_val - min_val
    base_variability = variability * value_range
    
    # Diverse initialization: three different strategies
    population = []
    
    # Strategy 1: Random uniform (1/3 of population)
    for _ in range(population_size // 3):
        S = min_val + rng.random(size=(n, n)) * value_range
        population.append(ensure_spd(S, min_eig=1e-4))
    
    # Strategy 2: Q-based with small perturbations (1/3)
    for _ in range(population_size // 3):
        noise_level = base_variability * 0.15
        S = Q + rng.normal(0, noise_level, size=(n, n))
        population.append(ensure_spd(S, min_eig=1e-4))
    
    # Strategy 3: Scaled identity + noise (remaining)
    for _ in range(population_size - 2 * (population_size // 3)):
        scale = 10.0 ** rng.uniform(-1, 1.5)  # Log-uniform from 0.1 to 30
        S = scale * jnp.eye(n) + rng.normal(0, base_variability * 0.08, size=(n, n))
        population.append(ensure_spd(S, min_eig=1e-4))
    
    num_elites = max(2, int(population_size * elite_fraction))
    best_fitness_history = []
    diversity_history = []
    stagnation_count = 0
    
    for gen in range(generations):
        # Evaluate all individuals
        fitnesses = jnp.array([calculate_accuracy(A, B, Q, R, S) for S in population])
        best_fitness = jnp.min(fitnesses)
        best_fitness_history.append(float(best_fitness))
        
        # Compute population diversity
        sorted_fit = jnp.sort(fitnesses)
        diversity = float(jnp.mean(sorted_fit[-int(len(sorted_fit) * 0.3):]) - best_fitness)
        diversity_history.append(diversity)
        
        # Adaptive annealing schedule
        progress = min(gen / generations, 1.0)
        if progress < 0.2:
            # Aggressive exploration phase
            current_variability = base_variability * 1.4
        elif progress < 0.5:
            # Transition to exploitation
            current_variability = base_variability * (1.4 - 0.6 * (progress - 0.2) / 0.3)
        elif progress < 0.8:
            # Steady exploitation
            current_variability = base_variability * 0.8
        else:
            # Fine-tuning phase
            current_variability = base_variability * (0.8 - 0.6 * (progress - 0.8) / 0.2)
        
        # Stagnation detection: check if diversity is dying
        if len(diversity_history) > 200:
            avg_recent_diversity = jnp.mean(jnp.array(diversity_history[-100:]))
            avg_old_diversity = jnp.mean(jnp.array(diversity_history[-200:-100]))
            if avg_recent_diversity < avg_old_diversity * 0.5:
                stagnation_count += 1
            else:
                stagnation_count = max(0, stagnation_count - 1)
        
        if stagnation_count > 25:  # Stop if diversity collapses
            break
        
        # Elite preservation
        sorted_indices = jnp.argsort(fitnesses)
        elite_indices = sorted_indices[:num_elites]
        elite_solutions = [population[int(i)] for i in elite_indices]
        
        # Tournament selection for breeding
        def tournament_select():
            candidates = rng.choice(len(population), size=tournament_size, replace=False)
            best_idx = min(candidates, key=lambda i: fitnesses[i])
            return population[best_idx]
        
        # Create new population
        new_population = elite_solutions.copy()
        
        while len(new_population) < population_size:
            # Two parents via tournament
            parent1 = tournament_select()
            parent2 = tournament_select()
            
            # Blend crossover: randomized mix ratio
            alpha = rng.uniform(0.2, 0.8)
            child = alpha * parent1 + (1 - alpha) * parent2
            
            # Adaptive mutation magnitude
            r = rng.uniform(0, 1)
            if r < 0.3:
                # Stronger mutation for diversity (30% chance)
                mutation_mag = current_variability * 1.5
            else:
                # Standard mutation
                mutation_mag = current_variability
            
            child = mutate(child, mutation_mag, rng)
            child = ensure_spd(child, min_eig=1e-8)
            new_population.append(child)
        
        population = new_population[:population_size]
    
    # Final selection: return best solution found
    final_fitnesses = jnp.array([calculate_accuracy(A, B, Q, R, S) for S in population])
    best_idx = int(jnp.argmin(final_fitnesses))
    S = population[best_idx]
    return (deriveK(S, R, B), S)

## Evaluation of the approaches

Approaches are evaluated based on the time to reach a solution and how close it is according to the criterion $\sum (Q + SA + A^TS - SBR^{-1}B^TS)^2$.

In [ ]:
A = jnp.array([
    [0., 0., 1., 0.],
    [0., 0., 0., 1.],
    [0., -5.0 / 7.0 * 10, 0., 0.],
    [0., 0., 0., 0.],
])

B = jnp.array([
    [0.],
    [0.],
    [0.],
    [1.],
])

Q = jnp.diag(jnp.array([1200.0, 320.0, 480.0, 260.0]))
R = jnp.array([[0.01]])

def print_results(func):
    start_time = time.time()
    (K, S) = func(A, B, Q, R)
    print(f'Time elapsed: {time.time() - start_time}')
    print(f'Accuracy: {calculate_accuracy(A, B, Q, R, S):e}')
    print(f'Matrix equation result: {calculate_result(A, B, Q, R, S)}')
    print(f'K={K},\nS={S}')


In [ ]:

print('Pydrake solver: ')
print_results(LinearQuadraticRegulator)


In [ ]:
print('GA solver (SPD-constrained, diversity-preserving):')
print_results(solve_lqr_ga)

print('\n\nGA solver (aggressive with larger population):')
def solve_lqr_ga_aggressive(A, B, Q, R):
    return solve_lqr_ga(A, B, Q, R, 
                        population_size=768,
                        generations=6000,
                        variability=0.25,
                        elite_fraction=0.12,
                        tournament_size=5)
print_results(solve_lqr_ga_aggressive)